In [22]:
import shutil
import os

# Source directory containing CSV files
src_dir = '../GraphDataset'

# Destination directory
dest_dir = '.'

# Create destination directory if it doesn't exist
os.makedirs(dest_dir, exist_ok=True)

# List of CSV files to copy
csv_files = ['author.csv', 'paper.csv', 'organization.csv', 'paper_author.csv', 'paper_organization.csv']

# Copy each CSV file
for file in csv_files:
    src_path = os.path.join(src_dir, file)
    dest_path = os.path.join(dest_dir, file)
    if os.path.exists(src_path):
        shutil.copy2(src_path, dest_path)
    else:
        print(f"Warning: Source file {src_path} not found")


In [23]:
import pandas as pd

# Load CSVs
author_df = pd.read_csv('author.csv')
paper_df = pd.read_csv('paper.csv') 
organization_df = pd.read_csv('organization.csv')

# Add type columns and rename columns
author_df['type'] = 'author'
paper_df['type'] = 'paper'
organization_df['type'] = 'organization'

author_df = author_df.rename(columns={'node_id': 'Id', 'name': 'Label'})
paper_df = paper_df.rename(columns={'node_id': 'Id', 'title': 'Label'})
organization_df = organization_df.rename(columns={'node_id': 'Id', 'name': 'Label'})

# Save modified CSVs
author_df.to_csv('author.csv', index=False)
paper_df.to_csv('paper.csv', index=False)
organization_df.to_csv('organization.csv', index=False)


In [24]:
# Load paper-organization and paper-author relationships
paper_org_rel = pd.read_csv('paper_organization.csv')
paper_author_rel = pd.read_csv('paper_author.csv')

# Rename columns to Source,Target
paper_org_rel.columns = ['Source', 'Target'] 
paper_author_rel.columns = ['Source', 'Target']

# Save modified CSVs
paper_org_rel.to_csv('paper_organization.csv', index=False)
paper_author_rel.to_csv('paper_author.csv', index=False)


In [25]:
# Load relationships first to analyze paper degrees
paper_author_rel = pd.read_csv('paper_author.csv')
paper_org_rel = pd.read_csv('paper_organization.csv')

# Calculate degree (number of authors + organizations) for each paper
paper_author_counts = paper_author_rel.groupby('Source').size()
paper_org_counts = paper_org_rel.groupby('Source').size()
paper_degrees = (paper_author_counts + paper_org_counts.fillna(0)).sort_values(ascending=False)

# Get top 50 papers by degree
high_degree_papers = paper_degrees.head(5).index.tolist()

# Randomly sample 250 additional papers
remaining_papers = list(set(paper_df['Id']) - set(high_degree_papers))
sampled_remaining = pd.Series(remaining_papers).sample(n=min(300-5, len(remaining_papers)), random_state=42)

# Combine high degree and randomly sampled papers
paper_ids_to_keep = set(high_degree_papers + sampled_remaining.tolist())
sampled_papers = paper_df[paper_df['Id'].isin(paper_ids_to_keep)]

# Filter paper_author and paper_organization relationships
paper_author_rel = paper_author_rel[paper_author_rel['Source'].isin(paper_ids_to_keep)]
paper_org_rel = paper_org_rel[paper_org_rel['Source'].isin(paper_ids_to_keep)]

# Get authors and organizations to keep
authors_to_keep = set(paper_author_rel['Target'])
orgs_to_keep = set(paper_org_rel['Target'])

# Filter main dataframes
paper_df = sampled_papers
author_df = author_df[author_df['Id'].isin(authors_to_keep)]
organization_df = organization_df[organization_df['Id'].isin(orgs_to_keep)]

# Save filtered CSVs
paper_df.to_csv('paper.csv', index=False)
author_df.to_csv('author.csv', index=False)
organization_df.to_csv('organization.csv', index=False)
paper_author_rel.to_csv('paper_author.csv', index=False)
paper_org_rel.to_csv('paper_organization.csv', index=False)
